# Full Dataset Training Notebook

Upload this notebook, a built `neuroseg_2d-*.whl`, and your dataset archive to the notebook environment. Use a CUDA-enabled PyTorch kernel.

Fill every `<...>` path in the configuration cell before running cells in order. No dataset is downloaded automatically. Keep the wheel's full filename.

For fresh training, leave `RESUME_CHECKPOINT = None`. To resume, upload or extract the complete previous run and set its `latest.pt` path. `EPOCHS` means additional epochs when resuming; the optimizer restores its saved learning rate.

> Remote storage may be temporary. Download the final checkpoint archive before the runtime stops.

## 1. Imports and configuration

In [ ]:
import json
import shutil
from pathlib import Path

from IPython.display import FileLink, display

# Fill local paths after uploading or staging the required files.
ROOT = Path("<notebook-directory>")
WHEEL = Path("<wheel-file.whl>")
ARCHIVE = Path("<dataset-archive.zip>")
RAW_DIR = ROOT / "data" / "raw"
CHECKPOINTS_DIR = ROOT / "checkpoints"

# Choose fresh training or additional epochs from a complete previous run.
EPOCHS = 10
LEARNING_RATE = 1e-4
BATCH_SIZE = 16
RESUME_CHECKPOINT = None  # Or CHECKPOINTS_DIR / "runs" / "<run>" / "latest.pt".

## 2. Verify GPU and install package

In [ ]:
# Inspect GPU before installing the uploaded package.
!nvidia-smi

# Install dependencies and uploaded wheel without replacing environment PyTorch.
%pip install matplotlib nibabel numpy scikit-learn tqdm
%pip install --no-deps --force-reinstall "{WHEEL}"

# Verify package imports and CUDA.
import neuroseg_2d
import torch

assert torch.cuda.is_available(), "CUDA GPU is not available"
print(f"Package: {neuroseg_2d.__file__}")
print(f"GPU: {torch.cuda.get_device_name(0)}")

## 3. Extract and inspect dataset

Run once for a new dataset. Training applies fixed-crop preprocessing online when each raw patient enters the Dataset cache.

In [ ]:
# Extract raw pairs and verify one online-preprocessed batch.
print("Starting dataset extraction and preview. Do not rerun this cell.")

!python -u -m neuroseg_2d.data \
    --archive "{ARCHIVE}" \
    --raw-dir "{RAW_DIR}"

print("Dataset extraction and preview complete.")

## 4. Train full dataset

Wait until extraction and preview finish. A `[*]` beside a cell means it is still running; do not rerun it.

In [ ]:
# Train from scratch or resume latest state in a copied timestamped run.
resume_checkpoint_arg = (
    "" if RESUME_CHECKPOINT is None else f'--resume-checkpoint "{RESUME_CHECKPOINT}"'
)
print("Starting full-dataset training. Do not rerun this cell.")

!python -u -m neuroseg_2d.train \
    --data-dir "{RAW_DIR}" \
    --epochs {EPOCHS} \
    --learning-rate {LEARNING_RATE} \
    --batch-size {BATCH_SIZE} \
    --checkpoints-dir "{CHECKPOINTS_DIR}" \
    {resume_checkpoint_arg}

print("Full-dataset training complete.")

## 5. Review validation results

Metric rows report separate slice and patient scopes. Each scope keeps its total size and omits per-metric valid-value counters.

In [ ]:
# Load latest training report and metric history.
run_dirs = [path for path in (CHECKPOINTS_DIR / "runs").iterdir() if path.is_dir()]
run_dir = max(run_dirs, key=lambda path: path.name)
report = json.loads((run_dir / "run.json").read_text())
metrics_history = json.loads((run_dir / "metrics.json").read_text())

# Select validation metrics from best epoch.
best_epoch = report["best_epoch"]
best_metrics = next(row for row in metrics_history if row["epoch"] == best_epoch)

# Show best training result.
print(f"Run: {run_dir.name}")
print(f"Best epoch: {best_epoch}")
print(f"Best validation loss: {report['best_validation_loss']:.6f}")

# Check required recall and precision for both evaluation scopes.
for scope_name in ("slice", "patient"):
    scope_metrics = best_metrics[scope_name]
    recall = scope_metrics["recall"]
    precision = scope_metrics["precision"]
    print(
        f"{scope_name.title()}: recall={recall:.6f} "
        f"({'PASS' if recall > 0.95 else 'FAIL'}), "
        f"precision={precision:.6f} "
        f"({'PASS' if precision > 0.70 else 'FAIL'})"
    )

## 6. Package and download results

Run after training. Use the displayed link or the notebook file browser to save the archive.

In [ ]:
# Compress every run plus best/latest references into one downloadable archive.
archive_file = Path(
    shutil.make_archive(
        str(ROOT / f"neuroseg_checkpoints_{run_dir.name}"),
        "gztar",
        root_dir=CHECKPOINTS_DIR,
    )
)

print(f"Checkpoints: {CHECKPOINTS_DIR}")
print(f"Archive: {archive_file}")
display(FileLink(archive_file.relative_to(ROOT)))